In [39]:
import sys
sys.path.append('..')
from sqlalchemy import create_engine, text
import os

import geopandas as gpd
import pandas as pd
from geoalchemy2 import Geometry

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import API_AMIGOCLOUD_TOKEN_ADM
from config import POSTGRES_UTEA

POSTGRES_UTEA['DATABASE'] = 'utea_precision'

In [44]:
PATH_SHPS = RUTA_UNIDAD_ONE_DRIVE + r"/Ingenio Azucarero Guabira S.A/UTEA - SEMANAL - SIEMBRA/SIEMBRA CON PRESICION/2026/SHP_PUNTOS"

In [45]:
def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

def obtener_lotes_pendientes():
    engine = obtener_engine()
    # 1. Query para traer solo los registros con segmentos = FALSE
    query = text("""
        SELECT unidad_01, unidad_02, unidad_05, campanha 
        FROM siembra_surcado.data_lote 
        WHERE puntos_cargados IS FALSE
    """)
    
    lista_formateada = []
    try:
        with engine.connect() as conn:
            resultados = conn.execute(query)
            
            for fila in resultados:
                # 2. Formatear cada fila: "u01_u02_u05_campaña"
                # Usamos str() para asegurar que los números se puedan concatenar
                registro_string = f"{fila.unidad_01}_{fila.unidad_02}_{fila.unidad_05}_{fila.campanha}"
                lista_formateada.append(registro_string)
        return lista_formateada
    except Exception as e:
        print(f"Error al recuperar datos: {e}")
        return []
    
def obtener_id_lote(conn, nombre_archivo):
    """Busca el ID de la tabla maestra usando el nombre del archivo."""
    try:
        partes = nombre_archivo.split('_')
        u01, u05 = int(partes[0]), partes[2]
        
        query = text("""
            SELECT id FROM siembra_surcado.data_lote 
            WHERE unidad_01 = :u1 AND unidad_05 = :u5
        """)
        res = conn.execute(query, {'u1': u01, 'u5': u05}).fetchone()
        return res[0] if res else None
    except (IndexError, ValueError):
        return None

def preparar_geodataframe_utm(ruta_shp, lote_id):
    gdf = gpd.read_file(ruta_shp)
    gdf['IsoTime'] = pd.to_datetime(gdf['IsoTime'])
    # Seleccionar y renombrar columnas
    gdf['lote_id'] = lote_id
    #gdf.columns = [c.lower() for c in gdf.columns]
    gdf.columns = gdf.columns.str.lower()
    gdf = gdf.rename(columns={'geometry': 'geom'})
    gdf = gdf.set_geometry('geom')
    gdf = gdf.to_crs(epsg=32720)
    columnas_db = ['lote_id', 'heading', 'distance', 'swathwidth', 
                   'tilltype', 'sectionid', 'elevation', 'isotime', 'vehiclspee', 'geom']
    gdf = gdf[columnas_db].copy()
    return gdf

def cargar_puntos_shps(ruta_carpeta):
    """Función principal que coordina la lectura y carga a Postgres."""
    engine = obtener_engine()
    pendientes = obtener_lotes_pendientes() # Tu función que filtra puntos_cargados IS FALSE
    
    for nombre_archivo in pendientes:
        ruta_shp = os.path.join(ruta_carpeta, f"{nombre_archivo}.shp")
        print(ruta_shp)
        
        if not os.path.exists(ruta_shp):
            print(f"⚠️ Archivo no encontrado: {nombre_archivo}")
            continue

        try:
            with engine.begin() as conn:
                # 1. Identificar el lote
                lote_id = obtener_id_lote(conn, nombre_archivo)
                
                if lote_id is None:
                    print(f"❌ No existe registro maestro para: {nombre_archivo}")
                    continue

                # 2. Preparar los datos
                df_final = preparar_geodataframe_utm(ruta_shp, lote_id)
                
                # 3. Insertar detalles
                df_final.to_postgis(
                    'detalles_lote_utm', 
                    conn, 
                    schema='siembra_surcado', 
                    if_exists='append'
                )
                
                # 4. Actualizar estado
                conn.execute(text("""
                    UPDATE siembra_surcado.data_lote 
                    SET puntos_cargados = TRUE 
                    WHERE id = :id
                """), {'id': lote_id})
                
                print(f"✅ {nombre_archivo}: {len(df_final)} puntos cargados.")
                
        except Exception as e:
            print(f"❌ Error procesando {nombre_archivo}: {str(e)}")

In [62]:
# --- Ejemplo de uso ---
lotes_falsos = obtener_lotes_pendientes()

print(f"Se encontraron {len(lotes_falsos)} registros con segmentos sin procesar:")
lotes_falsos

Se encontraron 1 registros con segmentos sin procesar:


['2307_CAMBERRA_A5_2026']

In [63]:
cargar_puntos_shps(PATH_SHPS)

G:\/Ingenio Azucarero Guabira S.A/UTEA - SEMANAL - SIEMBRA/SIEMBRA CON PRESICION/2026/SHP_PUNTOS\2307_CAMBERRA_A5_2026.shp
✅ 2307_CAMBERRA_A5_2026: 132375 puntos cargados.
